## **Install Required Libraries**

Installed all required libraries:


In [1]:
!pip install sentence-transformers rank-bm25 google-generativeai

## **Cell 1: Secure Environment Setup & Model Initialization**

This cell handles the security logic you requested and warms up the embedding and cross-encoder models.Imported all necessary modules:



In [9]:
import os
import getpass
import numpy as np
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import google.generativeai as genai

# Load environment variables from .env file
load_dotenv()

# Secure API Key handling
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    # This will prompt you for the key if it's not in your .env
    api_key = getpass.getpass("Enter your Google API Key: ")
    os.environ["GOOGLE_API_KEY"] = api_key

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# Initialize Models
# SBERT for semantic vector search
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Cross-Encoder for high-accuracy re-ranking
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# UPDATED: Using current 2026 stable model names to avoid 404
# 'gemini-2.5-flash' is the standard workhorse; 'gemini-3-flash-preview' is also an option.
MODEL_NAME = 'gemini-2.5-flash'
gemini_model = genai.GenerativeModel(MODEL_NAME)

print(f"Setup complete. Using model: {MODEL_NAME}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setup complete. Using model: gemini-2.5-flash


## **CELL 2 :Setup Gemini API**

Configured Gemini API using API key

Loaded model: gemini-2.5-flash

This model is used for:

Query expansion

Final answer generation

In [10]:
from dotenv import load_dotenv
import os

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

print("Setup complete.")

Setup complete.


## **CELL 3: Create Document Corpus**

defining a corpus that includes semantic overlaps and specific technical terms  to prove why Hybrid Search is better than just Dense search.

In [11]:
corpus = [
    "The attention mechanism in deep learning allows models to focus on specific segments of the input sequence.",
    "Transformers use self-attention mechanisms to weigh the significance of different parts of input data.",
    "Multi-head attention enables the model to jointly attend to information from different representation subspaces.",
    "Stochastic Gradient Descent (SGD) is a fundamental optimization technique for training neural networks.",
    "The AdamW optimizer improves upon Adam by decoupling weight decay from the gradient update.",
    "Weight decay is a regularization technique that adds a penalty to the loss function based on the size of weights.",
    "Backpropagation calculates the gradient of the loss function with respect to each weight by the chain rule.",
    "The ReLU activation function helps mitigate the vanishing gradient problem in deep neural architectures.",
    "Convolutional Neural Networks (CNNs) use spatial hierarchies to process data with grid-like topology.",
    "The 'Transformer-XL' architecture introduces recurrence to capture extremely long-term dependencies beyond fixed lengths."
]

## **CELL 4: Hybrid Retrieval (BM25 + SBERT + RRF)**

This implements the core retrieval engine using Reciprocal Rank Fusion.

In [12]:
class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k # Constant for RRF

        # Initialize BM25 (Keyword)
        tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized_corpus)

        # Initialize SBERT (Semantic)
        self.corpus_embeddings = sbert_model.encode(corpus, convert_to_tensor=True)

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        # 1. BM25 Ranking
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_ranks = np.argsort(bm25_scores)[::-1]

        # 2. SBERT Ranking
        query_emb = sbert_model.encode(query, convert_to_tensor=True)
        cos_scores = util.cos_sim(query_emb, self.corpus_embeddings)[0]
        sbert_ranks = np.argsort(cos_scores.cpu().numpy())[::-1]

        # 3. Reciprocal Rank Fusion (RRF)
        rrf_scores = {}
        for rank, idx in enumerate(bm25_ranks):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (self.k + rank + 1)
        for rank, idx in enumerate(sbert_ranks):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (self.k + rank + 1)

        # Sort indices by RRF score
        sorted_indices = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        results = []
        for idx, score in sorted_indices[:top_k]:
            results.append({
                "doc_id": idx,
                "rrf_score": score,
                "bm25_rank": list(bm25_ranks).index(idx) + 1,
                "sbert_rank": list(sbert_ranks).index(idx) + 1,
                "text": self.corpus[idx]
            })
        return results

retriever = HybridRetriever(corpus)

## Cell 5: Part 3 & 4 — Query Expansion (HyDE) & Re-Ranking

HyDE creates a "hallucinated" answer to better match the technical documents, while the Cross-Encoder cleans up the final results

In [13]:
def get_hyde_query(query: str) -> str:
    """Generates a hypothetical answer to expand the query."""
    prompt = f"Provide a brief, technical one-sentence explanation for: {query}"
    response = gemini_model.generate_content(prompt, generation_config={"temperature": 0.0})
    return f"{query} {response.text}"

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """Uses a Cross-Encoder to refine the ranking of candidates."""
    # We pair the ORIGINAL user query with each candidate text
    pairs = [[query, c['text']] for c in candidates]
    scores = cross_encoder.predict(pairs)

    for i, score in enumerate(scores):
        candidates[i]['cross_score'] = float(score)

    reranked = sorted(candidates, key=lambda x: x['cross_score'], reverse=True)
    return reranked[:top_k]

## Cell **6**: Part 5 — The Advanced RAG Pipeline

This connects every component into a single callable function.

In [14]:
def advanced_rag(user_query: str) -> str:
    # 1. Expand Query (HyDE)
    expanded_query = get_hyde_query(user_query)

    # 2. Retrieve (Hybrid)
    retrieved = retriever.retrieve(expanded_query, top_k=6)

    # 3. Re-Rank
    final_docs = rerank(user_query, retrieved, top_k=3)

    # 4. Generate Final Answer
    context_text = "\n".join([f"- {d['text']}" for d in final_docs])
    final_prompt = f"""You are a university AI assistant. Answer the question using ONLY the context provided.
    Context:
    {context_text}

    Question: {user_query}
    Answer:"""

    response = gemini_model.generate_content(final_prompt)
    return response.text

# Helper for comparison: Basic Semantic Search only
def naive_rag_top_doc(user_query: str) -> str:
    query_emb = sbert_model.encode(user_query, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, retriever.corpus_embeddings)[0]
    return corpus[np.argmax(scores.cpu().numpy())]

## Cell 7: Part 6 — Comparison Experiment

This cell to generate the data for your final table.

In [15]:
test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "details on Transformer-XL"
]

results_table = []

for q in test_queries:
    naive_doc = naive_rag_top_doc(q)

    # Run Advanced logic to see top doc after reranking
    expanded = get_hyde_query(q)
    retrieved = retriever.retrieve(expanded, top_k=6)
    adv_top_doc = rerank(q, retrieved, top_k=1)[0]['text']

    results_table.append({
        "Query": q,
        "Naive": naive_doc,
        "Advanced": adv_top_doc,
        "Different": "Yes" if naive_doc != adv_top_doc else "No"
    })

# Print for easy copy-pasting to Markdown
print(f"| {'Query':<35} | {'Naive RAG Top Doc':<35} | {'Adv RAG Top Doc':<35} | {'Different?'} |")
print("|" + "-"*37 + "|" + "-"*37 + "|" + "-"*37 + "|" + "-"*12 + "|")
for r in results_table:
    print(f"| {r['Query']:<35} | {r['Naive'][:33]:<35} | {r['Advanced'][:33]:<35} | {r['Different']:<10} |")

| Query                               | Naive RAG Top Doc                   | Adv RAG Top Doc                     | Different? |
|-------------------------------------|-------------------------------------|-------------------------------------|------------|
| how do transformers encode meaning? | Transformers use self-attention m   | Transformers use self-attention m   | No         |
| optimization techniques for training | Stochastic Gradient Descent (SGD)   | Stochastic Gradient Descent (SGD)   | No         |
| details on Transformer-XL           | The 'Transformer-XL' architecture   | The 'Transformer-XL' architecture   | No         |


| **Metric**      | **What it is**             | **What it Implies **                                                                                       |
| --------------- | -------------------------- | ------------------------------------------------------------------------------------------------------------------------ |
| **BM25 Rank**   | Keyword matching rank      | **High Rank (1–3):** The document contains the exact technical terms or proper nouns you typed.                          |
| **SBERT Rank**  | Semantic (meaning) rank    | **High Rank (1–3):** The document is related to the *topic* even if it doesn’t use the same words.                       |
| **RRF Score**   | Combined "Consensus" score | **High Score:** This document is the "winner" because both keyword and semantic search agreed it was important.          |
| **Cross-Score** | Deep Re-ranking score      | **Positive/High:** The Cross-Encoder confirms this document directly answers the user's question, filtering near-misses. |
| **HyDE Query**  | The "Imagined" answer      | **Detailed Text:** A textbook-like answer implies the retriever has a better "target" than a short, vague user query.    |
